# Spotiflow in Python

<div class="custom-button-row">
    <a 
        class="custom-button custom-download-button" href="../../notebooks/00_spot_detection/spotiflow_notebook.ipynb" download>
        <i class="fas fa-download"></i> Download this Notebook
    </a>
    <a
    class="custom-button custom-download-button" href="https://colab.research.google.com/github/bobiac/bobiac-book/blob/gh-pages/colab_notebooks/00_spot_detection/spotiflow_notebook_colab.ipynb" target="_blank">
        <img class="button-icon" src="../../_static/logo/icon-google-colab.svg" alt="Open in Colab">
        Open in Colab
    </a>
</div>

In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "matplotlib",
#     "spotiflow",
#     "ndv[jupyter,pygfx]",
#     "jupyter-rfb<=0.5.4",
# ]
# ///

## Overview

[GitHub](https://github.com/weigertlab/spotiflow) | [Paper](https://www.nature.com/articles/s41592-025-02662-x) | [Spotiflow Documentation](https://weigertlab.org/spotiflow/index.html) | [Spotiflow API](https://weigertlab.org/spotiflow/api.html#)

...

In this notebook, we'll see how to run Spotiflow on single images or on a folder of images, and how to visualize and save the results.

The images we will use for this section can be downloaded from the <a href="" download> <i class="fas fa-download"></i> Spot Detection Dataset</a>.

<p class="alert alert-warning">
    <strong>💡 Tip:</strong> Spotiflow runs significantly faster on a GPU. It supports both NVIDIA GPUs (CUDA) and Apple Silicon (MPS). If you don't have either, we recommend running this notebook on <a href="https://colab.research.google.com/github/bobiac/bobiac-book/blob/gh-pages/colab_notebooks/00_spot_detection/spotiflow_notebook_colab.ipynb" target="_blank"> Google Colab</a> for faster performance.
</p>

<details>
<summary><b>NVIDIA GPU (CUDA - Windows/Linux)</b></summary>
<br>

In order to use Spotiflow in this notebook with an NVIDIA GPU:
1. you need to have the [NVIDIA drivers](https://www.nvidia.com/en-us/drivers/) installed on your system.
2. you can run `nvidia-smi` in the terminal to check your CUDA version (shown in the top-right of the output, e.g. `CUDA Version: 13.0.0`).
3. update the `# /// script` block at the top of this notebook to install the appropriate version of [PyTorch with CUDA support](https://pytorch.org/get-started/locally/) (replace `cu130` with your CUDA version):

```python
    # /// script
    # requires-python = ">=3.12"
    # dependencies = [
    #     "matplotlib",
    #     "spotiflow",
    #     "tqdm",
    #     "torch",
    #     "torchvision",
    # ]
    #
    # [tool.uv.sources]
    # torch = { index = "pytorch-cu130" }
    # torchvision = { index = "pytorch-cu130" }
    #
    # [[tool.uv.index]]
    # name = "pytorch-cu130"
    # url = "https://download.pytorch.org/whl/cu130"
    # explicit = true
    # ///
```

4. re-run the notebook using `uvx juv run`.
</details>

## Import Libraries

In [2]:
import csv

import matplotlib.pyplot as plt
import ndv
import numpy as np
import tifffile
from spotiflow.model import Spotiflow

## Setup

These are a few helper functions that we will use later:

- `save_points_as_csv`: save the detected spots as a `napari`-compatible `csv` file.
- `split_points_by_channel`: split the spots returned by `predict_multichannel()` into one array per channel.
- `plot_points_by_channel`: overlay the detected spots on each channel of the image with `matplotlib`.

In [ ]:
def save_points_as_csv(points, output_path="points.csv", channel_last=False) -> None:
    """Save points as a napari-compatible CSV (drag-and-drop as Points layer).

    napari maps the CSV columns (axis-0, axis-1, ...) to the layer axes in order.
    `predict_multichannel` returns the channel as the *last* column (e.g. (y, x, channel)),
    so set `channel_last=True` to move it to the front (e.g. (channel, y, x)) and have the
    spots line up with a channel-first (C, ...) image in napari.
    """
    points = np.asarray(points)
    if channel_last:
        # move the last column (channel) to the front
        points = points[:, [-1, *range(points.shape[1] - 1)]]
    ndim = points.shape[1]
    headers = ["index"] + [f"axis-{i}" for i in range(ndim)]
    with open(output_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        for i, p in enumerate(points):
            writer.writerow([i, *p])


def split_points_by_channel(points, n_channels):
    """Split the points from `predict_multichannel` into one array per channel.

    `points` has the channel index as its last column (e.g. (y, x, channel)).
    Returns a tuple of length `n_channels`, where each element is an array with
    the spatial coordinates only (channel column dropped) of the spots detected
    in that channel, ready to be overlaid with matplotlib.
    """
    points = np.asarray(points)
    channels = points[:, -1].astype(int)
    coords = points[:, :-1]
    return tuple(coords[channels == c] for c in range(n_channels))


def plot_points_by_channel(image, points, colors=None):
    """Overlay the spots from `predict_multichannel` on each channel of an image.

    `image` is channel-first (C, Y, X) and `points` has the channel index as its
    last column. Shows one subplot per channel with that channel's spots overlaid.
    """
    n_channels = image.shape[0]
    points_per_channel = split_points_by_channel(points, n_channels)
    fig, axs = plt.subplots(1, n_channels, figsize=(10, 5))
    axs = np.atleast_1d(axs)  # keep iterable when there is a single channel
    for ch, ax in enumerate(axs):
        spots = points_per_channel[ch]
        color = colors[ch] if colors is not None else "red"
        ax.imshow(image[ch], cmap="gray")
        ax.scatter(spots[:, 1], spots[:, 0], s=20, edgecolor=color, facecolor="none")
        ax.set_title(f"channel {ch} ({len(spots)} spots)")
        ax.axis("off")
    plt.show()

## Running Spotiflow on 2D images

### Load the Image

Since we will be using TIFF files, to load the images, we can use the `imread` method from the `tifffile` library.

In [4]:
image_path = "../../_static/images/spots/2d_2ch_spots.tif"
image = tifffile.imread(image_path)

print(image.shape)

(2, 256, 256)


This is a 2-channel image, we can use the `imshow` method from the `ndv` library to visualize the image.

In [6]:
ndv.imshow(
    image,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "magenta"}},
)

RFBOutputContext()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Initialize the Model

To initialize a pre-trained Spotiflow model, we can use the [`from_pretrained()`](https://weigertlab.org/spotiflow/api.html#spotiflow.model.spotiflow.Spotiflow.from_pretrained) class method.

Currently the available pre-trained models are:

- `general`: trained on a diverse dataset of spots of different modalities acquired in different microscopes with different settings.
- `hybiss`: trained on HybISS data acquired in 3 different microscopes
- `synth_complex`: trained on synthetic data, which includes simulations of aberrated spots and fluorescence background.
- `fluo_live`: trained on live-cell fluorescence imaging data corresponding to the Telomeres and Terra datasets in the manuscript.
- `synth_3d`: trained on synthetic 3D data, which includes simulations of aberrated spots and Z-related artifacts.
- `smfish_3d`: fine-tuned from the synth_3d model on smFISH 3D data of Platynereis dumerilii.

For this first example, we'll use the `general` model.

<p class="alert alert alert-info">
    <strong>Note:</strong> If you never used Spotiflow before, the model you will specify in the <i>from_pretrained()</i> method will be downloaded automatically the first time you run this notebook.
</p>

In [7]:
# Load a pretrained model
model = Spotiflow.from_pretrained("general")

INFO:spotiflow.model.spotiflow:Loading pretrained model: general


### Run Spotiflow

After initializing the model, we can run Spotiflow on the image using either the [`predict()`](https://weigertlab.org/spotiflow/api.html#spotiflow.model.spotiflow.Spotiflow.predict) or the [`predict_multichannel()`](https://weigertlab.org/spotiflow/api.html#spotiflow.model.spotiflow.Spotiflow.predict_multichannel) method from the initialized model.

The difference between the two is:

- `predict()` runs the model on a **single-channel** image (a 2D `(Y, X)` image or a 3D `(Z, Y, X)` stack) and returns the spots detected in it.
- `predict_multichannel()` is a convenience wrapper for **multi-channel** images: it runs `predict()` on **each channel independently** and stacks the results, tagging every spot with the channel it was detected in. This is needed because the pre-trained models always work on a single channel at a time.

In both cases, Spotiflow expects the **channel axis to be last**, i.e. the image must be in `(Y, X, C)` shape for 2D or `(Z, Y, X, C)` for 3D (in short, `(Z)YXC`). Multi-channel microscopy images, however, are more commonly stored **channel-first** as `(C, Y, X)` (or `(C, Z, Y, X)`), like the one we just loaded (its shape is `(2, 256, 256)`, i.e. `(C, Y, X)`). So before running the model we move the channel axis to the end with [`numpy.transpose`](https://numpy.org/doc/stable/reference/generated/numpy.transpose.html):

```python
# image.shape -> (C, Y, X) = (2, 256, 256)
image = np.transpose(image, (1, 2, 0))
# image.shape -> (Y, X, C) = (256, 256, 2)
```

The tuple `(1, 2, 0)` tells `np.transpose` the new order of the axes: take the original axis `1` (Y) first, then axis `2` (X), then axis `0` (C). For a 3D `(C, Z, Y, X)` stack you would instead use `np.transpose(image, (1, 2, 3, 0))` to obtain `(Z, Y, X, C)`.

Since our image has 2 channels, we'll use `predict_multichannel()`.

<p class="alert alert alert-info">
    <strong>💡 Tip:</strong> If you have a <b>single-channel</b> image you don't need the multichannel method (and no transpose): just call <code>predict()</code> directly on the image, e.g. <code>points, details = model.predict(image)</code>.
</p>

`predict_multichannel()` will return two outputs (a tuple):
- `points`: a numpy array containing the coordinates of the detected spots. Each row is `(y, x, channel)` for a 2D image (or `(z, y, x, channel)` for a 3D stack), where the **last column is the index of the channel** the spot was detected in.

- `details`: a **list** with one entry per channel, each containing the spot-wise details for that channel:
    - `heatmap`: the heatmap of the probabilities per pixel
    - `intens`: the spot intensities
    - `prob`: the probability of each detected spot to be a true positive
    - `flow`: the stereographic flow vector field
    - `subpix`: the 2D local offset vector field

<details>
<summary><b><code>predict_multichannel()</code> signature</b></summary>
<br>

The `predict_multichannel()` signature is not yet documented in the Spotiflow API reference, so we report it here:

```python
model.predict_multichannel(
    img,              # np.ndarray in channel-last format: (Y, X, C) or (Z, Y, X, C)
    channels=None,    # int or tuple of channel indices to run on; None runs on all channels
    **predict_kwargs, # any keyword argument accepted by predict(),
                      # e.g. prob_thresh, min_distance, scale, subpix, normalizer, device
)
```

It returns a tuple `(points, details)`:

- `points`: numpy array of shape `(N, 3)` for 2D images or `(N, 4)` for 3D stacks, containing the spot coordinates with the **channel index as the last column**.
- `details`: a list with one entry per processed channel, each holding that channel's `heatmap`, `intens`, `prob`, `flow` and `subpix`.

</details>


In [8]:
image1 = image.transpose(1, 2, 0)  # (C, Y, X) -> (Y, X, C)
print(image1.shape)

points, details = model.predict_multichannel(image1)

(256, 256, 2)
INFO:spotiflow.model.spotiflow:Data is assumed to be in channel-last format ((Z)YXC).


Predicting channels:   0%|          | 0/2 [00:00<?, ?it/s]

### Explore and Display the Results
Let's explore the outputs of the `predict()` method starting from understanding the `points`.

What is the `shape` of `points`?

In [9]:
print(points.shape)

(100, 3)


What are the coordinates of the first detected spot?

In [ ]:
print(points[0].shape)

Now let's visualize the detected spots on top of the image.

Since `predict_multichannel()` returns the spots of **all channels stacked together** (with the channel index in the last column), we use the `plot_points_by_channel` helper we defined in the [Setup](#Setup) section to overlay each channel's spots on the corresponding channel of the image (it internally splits the points per channel with `split_points_by_channel`).

In [ ]:
plot_points_by_channel(image, points, colors=["green", "magenta"])

We can also save the detected spots for future use as a `csv` file using the `save_points_as_csv` function we defined at the beginning of this notebook.

Since we used `predict_multichannel()`, each row of `points` is in `(y, x, channel)` format. napari maps the columns of a points `csv` to the layer axes **in order**, and our image is stored as `(C, Y, X)`, so the channel index must come **first**. We let the function take care of this by passing `channel_last=True`, which moves the channel column to the front (`(channel, y, x)`) before saving.

<p class="alert alert alert-info">
    <strong>Note:</strong> This function is compatible with <i>napari</i>, so you can directly drag and drop the saved <i>csv</i> file in <i>napari</i> (together with the original multi-channel image) to visualize the detected spots as a points layer: each spot will appear on the channel it was detected in.
</p>

In [ ]:
save_points_as_csv(points, "2d_points.csv", channel_last=True)

Let's have a look at the `details` output as well and let's focus in particular on the `heatmap` and the intensities (`intens`) of the detected spots.

We can first plot the `heatmap` of the probabilities per pixel by visualizing it with `imshow` from the `ndv` library so that we can interactively explore the values of the heatmap by hovering over it with the mouse cursor.

In [ ]:
ndv.imshow(details.heatmap, default_lut={"cmap": "hot"})

In [ ]:
viewer = ndv.imshow(details.heatmap, default_lut={"cmap": "hot"})

In [ ]:
viewer.widget().children[1].snapshot()

In `details.intens` we can find the value of the pixel in the original image at the coordinates of each detected spot, which can be used as a measure of the intensity of the detected spots.

We can visualize the distribution of these intensities with a histogram using `hist` from `matplotlib.pyplot`.

In [ ]:
plt.hist(details.intens, bins=15)
plt.xlabel("Intensity")
plt.ylabel("Frequency")
plt.show()

## Running Spitiflow on a Folder of Images

## Running Spitiflow on 3D images

### Load the Image

In [ ]:
image_path = "../../_static/images/spots/3d_spots.tif"
image_3d = tifffile.imread(image_path)

print(image_3d.shape)

In [ ]:
ndv.imshow(image_3d)

In [ ]:
# Load a pretrained model
model = Spotiflow.from_pretrained("smfish_3d")

In [ ]:
points, details = model.predict(image_3d)

In [ ]:
points[0]

In [ ]:
# save_points_as_csv(points, "points_3d.csv")
save_points_as_csv(points, "../../../_static/images/spots/3d_spots.csv")

In [ ]:
ndv.imshow(details.heatmap, default_lut={"cmap": "hot"})

In [ ]:
# img = test_image_hybiss_2d()

In [ ]:
# from skimage.feature import blob_log, peak_local_max
# from skimage.filters import gaussian

In [ ]:
# # --- simple local-maxima ("find maxima") ---
# # peak_local_max is the direct analog of ImageJ's "Find Maxima." Big-FISH / FISH-quant essentially wrap LoG filtering + local maxima + a fitted threshold.
# smoothed = gaussian(img, sigma=1)
# coords = peak_local_max(
#     smoothed,
#     min_distance=3,
#     # threshold_abs=0.02
#     threshold_rel=0.61,
# )

# print(len(coords), "spots detected")
# plt.imshow(img, cmap="gray")
# plt.scatter(coords[:, 1], coords[:, 0], s=30, edgecolor="g", facecolor="none")
# plt.axis("off")
# plt.show()

In [ ]:
# # --- LoG blob detection (handles spot size/scale) ---
# # blobs: array of (row, col, sigma); radius ≈ sqrt(2)*sigma
# blobs = blob_log(
#     img,
#     min_sigma=1,
#     max_sigma=10,
#     num_sigma=10,
#     threshold=0.001,
# )

# print(len(blobs), "spots detected")
# plt.imshow(img, cmap="gray")
# plt.scatter(blobs[:, 1], blobs[:, 0], s=30, edgecolor="g", facecolor="none")
# plt.axis("off")
# plt.show()

In [ ]:
# import tifffile

# img3d = tifffile.imread(
#     "/Users/fdrgsp/Documents/git/bobiac-book/_internal/3d_spots.tif"
# )

In [ ]:
# ndv.imshow(img3d)

In [ ]:
# img3d_smooth = gaussian(img3d, sigma=1)
# ndv.imshow(img3d_smooth.astype("float32"))

In [ ]:
# scale_x, scale_y, scale_z = 0.1, 0.1, 0.5
# anisotropy = scale_z / scale_x
# print("Voxel anisotropy (z/xy):", anisotropy)

# # peak_local_max keeps a pixel only if it is the maximum within this footprint,
# # so the footprint sets the *minimum separation* between detected peaks (it does
# # NOT change which spot sizes can be detected). For a 3D anisotropic stack we want
# # the footprint to span the same *physical* distance in every axis.
# #
# # CASE 1 - a physically round object (bead, nucleus): same size in µm in all axes.
# # Since z-voxels are `anisotropy` times larger, the object spans `anisotropy` times
# # fewer voxels in z, so the footprint must be `anisotropy` times WIDER in xy:
# #     footprint_xy / footprint_z = anisotropy            (= 5 here)
# #
# # CASE 2 - a diffraction-limited PSF spot (our data): the PSF is itself elongated
# # in z (sigma_z is ~psf_z_elong times larger than sigma_xy in µm). That partially
# # cancels the voxel anisotropy, so the spot is only ~1.75x wider in xy, NOT 5x:
# #     footprint_xy / footprint_z = anisotropy / psf_z_elong = 5 / 2.86 ≈ 1.75

# # PSF z-elongation from the Gaussian-PSF approximation (NA=0.75, RI=1, em=0.52 µm)
# na, ri, em = 0.75, 1.0, 0.520
# psf_z_elong = (0.45 * em * ri / na**2) / (0.21 * em / na)  # ≈ 2.86
# xy_over_z = anisotropy / psf_z_elong  # ≈ 1.75
# print(
#     "PSF z-elongation:",
#     round(psf_z_elong, 2),
#     "-> footprint xy/z:",
#     round(xy_over_z, 2),
# )

# # size the footprint to the spot's z-extent in voxels (measured ~9 here),
# # then make xy `xy_over_z` times wider
# n = 9
# footprint = np.ones((n, round(n * xy_over_z), round(n * xy_over_z)), dtype=bool)
# print("Footprint shape (z, y, x):", footprint.shape)

# coords = peak_local_max(
#     img3d_smooth,
#     footprint=footprint,
#     threshold_rel=0.2,
#     # threshold_abs=0.1
# )
# print(len(coords), "spots detected")

In [ ]:
# # For anisotropic voxels you pass blob_log a per-axis sigma tuple (z, y, x).
# # Same correction as the footprint above: the z-sigma is NOT xy_sigma / anisotropy
# # (that assumes a physically round object). For a diffraction-limited PSF the spot
# # is z-elongated, so the voxel ratio is xy_over_z = anisotropy / psf_z_elong ≈ 1.75
# # (psf_z_elong and xy_over_z were computed in the peak_local_max cell above).
# # img3d is a 3D z-stack (Z, Y, X)

# # xy-sigma range to scan, in xy voxels (smallest..largest spot)
# min_xy, max_xy = 1.5, 6
# min_sigma = max_sigma = (min_xy / xy_over_z, min_xy, min_xy)  # z uses PSF ratio

# blobs = blob_log(
#     img3d_smooth,
#     min_sigma=min_sigma,
#     max_sigma=max_sigma,
#     num_sigma=10,
#     threshold_rel=0.05,  # 8% of max blob intensity (lower = more spots, higher = fewer spots
# )
# print(len(blobs), "spots detected")